# 02 Conservative speaker mapping

Code-switch counts use CHAT/TSV speaker codes, whereas education comes from questionnaire IDs. Linking them incorrectly would attach one person's education to another person's speech.

## 1. Load both inventories

The match evidence is restricted to fields actually present in the supplied files: normalised soundfile, age at recording and sex.

In [ ]:
from pathlib import Path
import collections, sys
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from bangor_miami.chat import chat_inventory
from bangor_miami.metadata import read_metadata
from bangor_miami.mapping import audit_mapping
chat_rows = chat_inventory(ROOT / 'data/sample/chats')
metadata_rows = read_metadata(ROOT / 'data/sample/sample_metadata.csv')
len(chat_rows), len(metadata_rows)


## 2. Build an auditable crosswalk

A row is confirmed only when exactly one questionnaire candidate matches all three evidence fields. Filename-only and elimination-based matches remain unresolved.

In [ ]:
mapping_rows = audit_mapping(chat_rows, metadata_rows)
status_counts = collections.Counter(row['mapping_status'] for row in mapping_rows)
{'status_counts': dict(status_counts), 'audit_rows': mapping_rows}


## 3. Inspect only confirmed links

Education is added downstream only for these rows; unresolved speakers never inherit guessed metadata.

In [ ]:
confirmed = [row for row in mapping_rows if row['mapping_status'].startswith('confirmed')]
[(row['chat_file'], row['chat_speaker_id'], row['questionnaire_id'], row['education_group']) for row in confirmed]


## Verified private-corpus audit

The strict rule confirmed 74 recording-speaker mappings representing 73 independent questionnaire participants. All other occurrences remained visible in the audit instead of being forced.